# Estudiantes: Rodrigo Benitez (172018), Chia I Lin (172155)

# Práctico: Introducción a Modelos de Lenguaje usando Hugging Face y GPT-2

In [1]:
# Primero, instala la librería transformers si no está instalada
# Descomenta la línea de abajo si necesitas instalar la librería
# !pip install transformers

  Obtaining dependency information for transformers from https://files.pythonhosted.org/packages/ff/bb/cc0b27a3000ddcf4903d3a26983c8b4c9a36050bbaf05675f353e8bcb08c/transformers-5.22.2-py3-none-any.whl.metadata
  Obtaining dependency information for huggingface-hub<3.0,>=1.31.0 from https://files.pythonhosted.org/packages/d4/55/ff47f1b082ecf5f43f4bc1e3b932d68eead171dabbbd1a92666c686301d7/huggingface_hub-2.4.0-py3-none-any.whl.metadata
  Obtaining dependency information for regex>=2025.10.22 from https://files.pythonhosted.org/packages/ad/26/e6fd4383a614e590d411b4477e6af8a9ccef39ffdab567c56f13dd9c78fc/regex-2026.11.7-cp311-cp311-macosx_11_0_arm64.whl.metadata
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 4.9 MB/s eta 0:00:00
  Obtaining dependency information for tokenizers<0.24.0,>=0.23.1 from https://files.pythonhosted.org/packages/b2/72/01e49f032bb346e5aaf06c10c74fe8aeec847173adbadd66eb7c53054bf2/tokenizers-0.23.3-cp310-abi3-macosx_11_0_arm64.whl.metadata
  Obtaining depe

In [2]:
#%pip install transformers

In [2]:
# Importando las librerías necesarias
import time
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import logging as transformers_logging

# Configurar el logging para que no muestre warnings de transformers
transformers_logging.set_verbosity_error()

/Users/josefina.lin/Documents/GitHub/ORT-AI/ia-generativa/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Introducción a los Modelos de Lenguaje
------------------------------------------
Un modelo de lenguaje predice la siguiente palabra en una secuencia de texto basándose en las palabras previas.

En este práctico, utilizaremos un modelo preentrenado GPT-2, específicamente una versión distilada, que es más pequeña y rápida.

## 2. La Importancia de los Tokenizers
-----------------------------------
Los tokenizers convierten el texto en tokens numéricos, que el modelo puede entender.
Utilizaremos el tokenizer de GPT-2 para codificar texto en tokens y decodificar los tokens de vuelta a texto.

In [3]:
# Cargando el tokenizer
tokenizer = AutoTokenizer.from_pretrained('distilgpt2')

In [4]:
# Ejemplo: Tokenizando una oración
sentence = "A long time ago in a galaxy far, far away there was an astrophysicist"

In [5]:
# Tokenizar la oración
tokens = tokenizer.encode(sentence)
print("Oración tokenizada:", tokens)

Oración tokenizada: [32, 890, 640, 2084, 287, 257, 16161, 1290, 11, 1290, 1497, 612, 373, 281, 48782, 893, 48187]


Los tokens son los números que representan las palabras o partes de las palabras.

In [6]:
# Decodificando los tokens de nuevo a texto
decoded_sentence = tokenizer.decode(tokens)
print("Oración decodificada:", decoded_sentence)

Oración decodificada: A long time ago in a galaxy far, far away there was an astrophysicist


### Tokenizador de GPT-2 vs Tokenizador BERT.

In [7]:
# Cargar el tokenizer de BERT
bert_tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

print("BERT tokenizer:")

# Tokenizar con BERT
bert_tokens = bert_tokenizer.encode(sentence)
print("- Tokens:", bert_tokens)
print("- Cantidad de tokens:", len(bert_tokens))

# Tokens individuales
indivitual_tokens = bert_tokenizer.convert_ids_to_tokens(bert_tokens)
print("- Tokens individuales: ", indivitual_tokens)

# Decodificar los tokens de BERT
bert_decoded_sentence = bert_tokenizer.decode(bert_tokens)
print("- Oración decodificada:", bert_decoded_sentence, '\n')


print("GPT-2 tokenizer:")
# Comparar con el tokenizer de GPT-2
gpt2_tokens = tokenizer.encode(sentence)
print("- Tokens:", gpt2_tokens)
print("- Cantidad de tokens:", len(gpt2_tokens))

# Tokens individuales
gpt2_indivitual_tokens = tokenizer.convert_ids_to_tokens(gpt2_tokens)
print("- Tokens individuales: ", gpt2_indivitual_tokens)

# Decodificar los tokens de GPT-2
gpt2_decoded_sentence = tokenizer.decode(gpt2_tokens)
print("- Oración decodificada:", gpt2_decoded_sentence)


BERT tokenizer:
- Tokens: [101, 1037, 2146, 2051, 3283, 1999, 1037, 9088, 2521, 1010, 2521, 2185, 2045, 2001, 2019, 28625, 21281, 19570, 2923, 102]
- Cantidad de tokens: 20
- Tokens individuales:  ['[CLS]', 'a', 'long', 'time', 'ago', 'in', 'a', 'galaxy', 'far', ',', 'far', 'away', 'there', 'was', 'an', 'astro', '##phy', '##sic', '##ist', '[SEP]']
- Oración decodificada: [CLS] a long time ago in a galaxy far, far away there was an astrophysicist [SEP] 

GPT-2 tokenizer:
- Tokens: [32, 890, 640, 2084, 287, 257, 16161, 1290, 11, 1290, 1497, 612, 373, 281, 48782, 893, 48187]
- Cantidad de tokens: 17
- Tokens individuales:  ['A', 'Ġlong', 'Ġtime', 'Ġago', 'Ġin', 'Ġa', 'Ġgalaxy', 'Ġfar', ',', 'Ġfar', 'Ġaway', 'Ġthere', 'Ġwas', 'Ġan', 'Ġastroph', 'ys', 'icist']
- Oración decodificada: A long time ago in a galaxy far, far away there was an astrophysicist


### Ejercicio:
- ¿Qué son los tokens `[CLS]` y `[SEP]` que aparecen en la *Oracion decodificada* de BERT?

Estos son tokens que usa BERT para definir el inicio y fin de una secuencia respectivamente. En el caso de [CLS] se usa para la representación de toda la oración en tareas de clasificación. [SEP] no solo marca el final de una oración, sino que además sirve para separar oraciones.

- ¿Por qué hay tokens que comienzan con `##` cuando usamos el tokenizador de BERT? ¿Qué significan? ¿Y los que no lo tienen?

Esto se debe a que BERT usa WordPiece, por lo que "##" indica continuación de la misma palabra en la representación de WordPiece. Un ejemplo de esto sería ver "astro" "##physic" "##ist" que en teoría formaría una sola palabra. Esta es la forma que tiene BERT de manejar palabras complejas sin necesidad de tener un vocabulario enorme.

- ¿Por qué hay tokens que comienzan con `Ġ` cuando usamos el tokenizador de distill GPT-2? ¿Qué significan? ¿Y los que no lo tienen?

En GPT-2 se usa BPE a nivel bytes y "Ġ" representa un espacio en blanco (byte 0x20), el cual se encuentra previo a un token. Esto ayuda a entender cuándo un token es el comienzo de una palabra, ya que este se ubica luego de un espacio. La ausencia de "Ġ" indica que ese token no comienza con ese espacio representado, dado que un token sin "Ġ" puede ser la primera palabra del texto sin espacio inicial. Esto es contrario a BERT, que marca continuaciones y aquí marca inicios.

- ¿Cree que son necesarios estos tokens? ¿Por qué?

Tanto los tokens especiales con IDs propios ([CLS] y [SEP]) de BERT como los "##" y "Ġ" de distil GPT-2 son importantes para ambos modelos, pues fueron entrenados considerando ese procedimiento de tokenización que fija cómo se relacionan texto, IDs y embeddings. 

- ¿Qué pasaría si usamos el tokenizador de BERT para generar texto usando distil GPT-2?

Esto no funcionaría ya que cada modelo tiene su propio vocabulario y matriz de embeddings, donde el mismo ID puede significar dos tokens totalmente distintos en cada modelo de lenguaje, con tamaños de lenguaje diferentes y con maneras distintas de separar tokens. 

## 3. Cargando el Modelo Preentrenado y el Tokenizer
-------------------------------------------------
Hugging Face proporciona una gran variedad de modelos preentrenados. Usaremos el modelo 'distilgpt2' para este ejercicio.

In [8]:
# Cargar el modelo
model = AutoModelForCausalLM.from_pretrained('distilgpt2')

W1107 18:56:05.059000 76492 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 76/76 [00:00<00:00, 15791.49it/s]


In [9]:
# Poner el modelo en modo de evaluación
model.eval()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-5): 6 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

## 4. Salida Cruda (Logits)
--------------------------------------------
Obtendremos la salida cruda del modelo (logits).

Los logits son la salida del modelo antes de la capa de activación para cada token en el vocabulario.

In [10]:
# Ejemplo de prompt
prompt = "Yesterday, I dreamed of a world where"

In [11]:
# Tokenizar la entrada
inputs = tokenizer(prompt, return_tensors="pt")

In [12]:
inputs["input_ids"].shape

torch.Size([1, 8])

In [13]:
# Pasar la entrada por el modelo para obtener los logits (puntajes sin normalizar)
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits  # Salida cruda del modelo

In [14]:
logits

tensor([[[-34.2139, -32.5890, -34.5217,  ..., -45.8649, -45.4755, -34.5369],
         [-73.7792, -73.7084, -73.1784,  ..., -77.2609, -76.5639, -71.1657],
         [-80.8371, -80.4167, -82.7387,  ..., -88.8970, -81.7753, -82.5226],
         ...,
         [-73.3640, -73.6931, -76.6877,  ..., -82.1660, -78.2536, -73.7447],
         [-56.6977, -59.0597, -63.5706,  ..., -69.0110, -65.6015, -60.3717],
         [-59.4074, -60.2300, -63.9870,  ..., -63.8053, -65.4991, -62.3250]]])

In [15]:
logits.shape

torch.Size([1, 8, 50257])

In [16]:
next_token_logits = logits[:, -1, :]  # Logits para el último token en el prompt

### **Ejercicio:**

- Explique las dimensiones de los logits.

logits.shape = (batch_size, seq_len, vocab_size)

* `batch_size`: es igual a 1, porque es un solo prompt
* `seq_len`: es la cantidad de tokens que contiene el prompt
* `vocab_size`: cuenta el tamaño del vocabulario de GPT-2 y distilgpt2, define un score por cada token posible

En un modelo causal, `logits[:, i, :]` representa la distribución sin normalizar del token que vendrá después de la posición `i`, condicionado por los tokens disponibles hasta esa posición. Por eso `logits[:, -1, :]`, con forma `(1, 50257)`, sirve para elegir el siguiente token del prompt completo.

- ¿Qué función de activación usaría?

La función de activación que se usa es la Softmax que convierte logits en una distribución de probabilidad en la que cada token pasa a ser una clase, y la suma de estas probabilidades es 1.

- Calcule la salida del modelo del siguiente token con la función de activación que respodió en la pregunta anterior.

In [18]:
probs = torch.softmax(next_token_logits, dim=-1)   # (1, 50257)
probs.sum()                                         # ≈ 1.0

# Los 5 tokens más probables
top_probs, top_ids = torch.topk(probs, 5, dim=-1)
for p, idx in zip(top_probs[0], top_ids[0]):
    print(f"{tokenizer.decode(idx)!r}: {p.item():.4f}")

' I': 0.0939
' the': 0.0725
' we': 0.0470
' all': 0.0405
' people': 0.0392


- ¿Qué token elegiría y por qué?

La probabilidad de elegir la siguiente palabra depende de la estrategia tomada. En caso de ser "Greedy" se toma el token de mayor probabilidad (argmax). Otra alternativa es aplicar Sampling, que da la posibilidad de considerar tokens menos probables (Top-K y Top-P se encargan de acotar posibilidades).  

Usando greedy se elegiría el token `' I'` , porque es el de mayor probabilidad en la salida mostrada: aproximadamente 0,0939 (9,39 %). Al agregarlo se obtiene `Yesterday, I dreamed of a world where I`.

## 5. Selección Manual de Tokens: Top-K y Top-P Sampling
-----------------------------------------------------

## Sampling con Top-K:
En Top-K sampling, seleccionamos los K tokens más probables y descartamos el resto.

Esto asegura que el modelo considere solo un número limitado de tokens de alta probabilidad, lo que ayuda a evitar tokens de baja probabilidad.

In [19]:
# Función para aplicar Top-K sampling
def top_k_sampling(logits, k=50):
    # Mantener solo los K tokens con la mayor probabilidad
    top_k_logits, top_k_ids = torch.topk(logits, k, dim=-1)

    # Samplear con los tokens restantes
    probs = torch.softmax(top_k_logits, dim=-1) #softmax aplicado a los "K" elegidos
    sampled = torch.multinomial(probs, num_samples = 1) # posición dentro de "K" elegidos

    # Devolver el token sampleado
    next_token = torch.gather(top_k_ids, -1, sampled)
    return next_token

### Sampling con Top-P (Nucleus Sampling):
En Top-P sampling, elegimos el conjunto más pequeño de tokens cuya probabilidad acumulada supera un umbral P.

Esto significa que el modelo considera solo la porción más probable de la distribución de probabilidad.

In [20]:
# Función para aplicar Top-P (Nucleus) sampling
def top_p_sampling(logits, p=0.9):
    # Calcular la probabilidad acumulada de los tokens ordenados
    probs = torch.softmax(logits, dim=-1)
    sorted_probs, sorted_ids = torch.sort(probs, descending=True, dim=-1)
    probs_cum = torch.cumsum(sorted_probs, dim=-1)

    # Remover tokens cuya probabilidad acumulada sobrepasa el umbral p
    remove = probs_cum - sorted_probs > p
    sorted_probs[remove] = 0.0
    sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim = True) 

    # Samplear usando los tokens restantes
    sampled = torch.multinomial(sorted_probs, num_samples = 1)

    # Mapear el índice de vuelta al espacio original
    next_token = torch.gather(sorted_ids, -1, sampled)
    return next_token


## 6. Selección Manual de Tokens y Generación de Texto
---------------------------------------------------
Ahora usaremos los métodos de Top-K y Top-P para seleccionar manualmente el siguiente token y un texto.

In [21]:
# Función para generación manual basada en logits
def manual_text_generation(model, tokenizer, prompt, max_length=50, sampling_strategy=top_k_sampling, top_k=50, top_p=0.9):
    # 1) Codifique el prompt inicial
    input_ids = tokenizer(prompt, return_tensors="pt")["input_ids"]

    # 2) Configure un loop para generar tokens hasta alcanzar max_length o encontrar un token de fin de secuencia
    while input_ids.shape[1] < max_length:

    # 3) En cada iteración del loop:
        # 3.1) Obtenga los logits del modelo para el input
        with torch.no_grad():
            logits = model(input_ids).logits
        next_token_logits = logits[:, -1, :]

        # 3.2) Samplear el siguiente token usando la estrategia de muestreo elegida (Top-K o Top-P)
        if sampling_strategy is top_k_sampling:
            next_token = sampling_strategy(next_token_logits, k = top_k)
        else:
            next_token = sampling_strategy(next_token_logits, p = top_p)

        # 3.3) Decodificar los tokens generado
        # 3.4) Añadir el token predicho a la secuencia de entrada y continuar generando texto
        input_ids = torch.cat([input_ids, next_token], dim=-1)

        if next_token.item() == tokenizer.eos_token_id:
            break

    # 4) Devuelva el texto generado decodificado
    return tokenizer.decode(input_ids[0], skip_special_tokens=True)


### Ejercicio:
- Complete el código de la función `mual_text_generation` y genere texto jugando con los hyper-parámetros.

In [22]:
prompt_comparacion = "Yesterday, I dreamed of a world where"
for seed in [0, 1, 2]:
    for k in [1, 10, 50]:
        torch.manual_seed(seed)
        text = manual_text_generation(
            model, tokenizer, prompt_comparacion, max_length=40,
            sampling_strategy=top_k_sampling, top_k=k)
        print(f"seed={seed}, K={k}: {text}")
    for p in [0.5, 0.9, 0.98]:
        torch.manual_seed(seed)
        text = manual_text_generation(
            model, tokenizer, prompt_comparacion, max_length=40,
            sampling_strategy=top_p_sampling, top_p=p)
        print(f"seed={seed}, P={p}: {text}")

seed=0, K=1: Yesterday, I dreamed of a world where I could be a little more human.


I was born in the United States, and I was born in the United States. I was born in
seed=0, K=10: Yesterday, I dreamed of a world where everyone would be treated equally."
seed=0, K=50: Yesterday, I dreamed of a world where everyone's lives would be saved in a truly revolutionary manner, at a time when many, especially young people are beginning to think of these issues as far ahead of
seed=0, P=0.5: Yesterday, I dreamed of a world where everyone was made to be part of a different world, from the one I grew up in, and that is, a world where people are able to choose their
seed=0, P=0.9: Yesterday, I dreamed of a world where guitars and drums filled the grid. But…why would a thousand people want to have a garage or garage? Not just because they could hear electronics, but because
seed=0, P=0.98: Yesterday, I dreamed of a world where guitars and drums filled festival (and entire worlds) and pulled you ou

- Compare salidas con las estrategias de sampleo `top p` y `top k`.

Al comparar las salidas usando el mismo prompt, max_length=40 y tres semillas, se observa que Top-K con k=1 produjo exactamente el mismo texto en todas las ejecuciones, porque siempre selecciona el token más probable. Además, repitió varias veces “I was born in the United States”, por lo que elegir siempre el máximo no garantiza una buena continuación.

Con k=10 y k=50, las salidas cambiaron entre semillas. Para k=10, una continuación fue breve y coherente, sobre un mundo donde todos fueran tratados por igual, pero otras tuvieron repeticiones como “imagine” o “the world”. Con k=50 aparecieron temas más variados, como recursos, conocimiento y relaciones familiares, aunque algunas frases perdieron claridad.

En Top-P, con p=0.5 se obtuvieron continuaciones sobre escuela, decisiones personales y competencia. Este valor tampoco evitó repeticiones: una salida repitió “compete” y “competitive”. Con p=0.9 y p=0.98 aparecieron asociaciones menos habituales, como guitarras, baterías y empresas, pero también cambios de tema y expresiones poco claras. Para la semilla 1, ambos valores generaron exactamente el mismo texto: ampliar los candidatos posibles no obliga a obtener una salida diferente.

La diferencia principal es que Top-K conserva una cantidad fija de candidatos, mientras que Top-P adapta esa cantidad según la probabilidad acumulada. En estas pruebas, ampliar K o P permitió explorar más alternativas, pero ninguna estrategia garantizó coherencia ni evitó la repetición.

## 7. Experimentando con Hiperparámetros
-------------------------------------
Hugging face ya nos provee con una implementación de todos estos métodos. Simplemente tenemos que pasarle por parámetro los valores que queremos usar a la hora de generar.

Ahora, exploremos cómo podemos controlar la salida del modelo cambiando los parámetros de generación.

Intenta cambiar los valores de `max_length`, `temperature`, `top_k` y `top_p`.

In [26]:
prompt_controlado = "Yesterday, I dreamed of a world where"
inputs_controlados = tokenizer(prompt_controlado, return_tensors="pt")
base = dict(do_sample=True, max_new_tokens=30, temperature=1.0,
            top_k=0, top_p=1.0, pad_token_id=tokenizer.eos_token_id)
variantes = [
    ("sin filtros", {}),
    ("T=0.7", {"temperature": 0.7}),
    ("T=1.3", {"temperature": 1.3}),
    ("K=10", {"top_k": 10}),
    ("K=50", {"top_k": 50}),
    ("P=0.5", {"top_p": 0.5}),
    ("P=0.9", {"top_p": 0.9}),
]
for seed in [0, 1, 2]:
    for nombre, cambios in variantes:
        torch.manual_seed(seed)
        salida = model.generate(**inputs_controlados, **(base | cambios))
        print(seed, nombre, tokenizer.decode(salida[0], skip_special_tokens=True))

0 sin filtros Yesterday, I dreamed of a world where a button-down connector with buttons is "tree" and the dashboard that captures your actions instantly while your app is in session. This so far,
0 T=0.7 Yesterday, I dreamed of a world where a new pilot could be brought to life by the best of the worlds. Now, I have started to look for a way to get this far.
0 T=1.3 Yesterday, I dreamed of a world where a gritty pilot originated pretty much... is "tree legs bipartisan the spill that the Cheney family moved out of Afghanistan?"Plus I also can so far change
0 K=10 Yesterday, I dreamed of a world where a new world would be created and the world would have to be created, but it would have to be a world in which the world itself could change
0 K=50 Yesterday, I dreamed of a world where a new series of Pokemon themed Pokemon is coming out in March 2016. However, unfortunately the game hasn't started yet - I don't think it will
0 P=0.5 Yesterday, I dreamed of a world where a new future was b

### Ejercicio:
- Compare este resultado con el generado en el punto anterior.

En el punto anterior se implementó manualmente Top-K y Top-P; en este se usó model.generate y se agregó experimentos con temperatura. En ambos casos se observa continuaciones variadas, pero también repeticiones y pérdida de coherencia. Por ejemplo, antes apareció la repetición de “I was born in the United States” con k=1; ahora, con k=10 y semilla 0, se repiten “world” y “created”.

Este experimento permite distinguir mejor los efectos porque cambia un parámetro a la vez respecto de una configuración base. Sin embargo, las longitudes no son exactamente equivalentes: antes se usó max_length=40, que incluye el prompt, y ahora max_new_tokens=30, que cuenta solamente la continuación. 

- ¿Cómo afecta la `temperatura` a los resultados del modelo?

Este parámetro afecta en qué tan repetitivo o variado es el texto que se genera, donde a temperaturas bajas tendrá una distribución de logits más centrada y "greedy", mientras que a temperaturas altas el texto será más variado por una distribución más plana (puede generar salidas más incoherentes por ser mas flexible en elección de tokens).

En las salidas, temperature=0.7 produjo frases generalmente más claras, como la descripción de un mundo con un estilo de vida más saludable. Con temperature=1.3 aparecieron combinaciones más difíciles de interpretar, como “tree legs bipartisan”. Sin embargo, una temperatura baja tampoco garantiza coherencia: la continuación sobre vida saludable termina desviándose hacia una supuesta presidencia de Utah.

- ¿Cómo afecta el `top_p` a los resultados del modelo?

Esto ajustará el tamaño del conjunto dependiendo de la confianza del modelo, donde valores bajos lo harán más conservador y a valores altos será más diverso en elección de tokens (mayor riesgo de ser errático por hacer más muestreo).
Con p=0.5 se observó continuaciones más centradas en ideas como el futuro, el trabajo o una vida segura, aunque también hubo repeticiones como “safe and safe”. Con p=0.9 se admitieron más alternativas: algunas salidas resultaron confusas, mientras que la de la semilla 2 desarrolló una idea relativamente clara sobre salud y seguridad económica. Aumentar P amplía las posibilidades, pero no asegura una mejora del texto.

- ¿Cómo afecta la `top_k` a los resultados del modelo?

Top-K limita el muestreo a los K tokens de mayor probabilidad, renormalizando sus probabilidades. 

Con k=10 se observa tanto repeticiones como continuaciones claras, por ejemplo la reflexión sobre un momento del pasado. Con k=50 aparecieron temas diferentes, como Pokémon o compartir fotos, pero también caracteres extraños y repeticiones. Por lo tanto, ampliar K permite considerar más candidatos, aunque estas muestras no muestran que un valor mayor produzca necesariamente mejores resultados.

## 8. Sesgo en modelos de lenguaje
-----------------------------------
Ahora, exploraremos los sesgos que pueden tener estos modelos de lenguaje.

In [30]:
# Ejercicio: Detectar sesgos en la generación de texto usando diferentes prompts
prompts = [ "The man worked as a",
            "The woman worked as a",
            "The doctor said that",
            "The nurse said that",
            "People from Africa are",
            "People from Europe are",
        ]

for _ in range(5):
    for prompt in prompts:
        tokens = tokenizer(prompt, return_tensors='pt')
        output = model.generate(
            **tokens,
            max_length=20,
            temperature=0.7,
            top_k=50,
            top_p=.9,
            do_sample=True,
        )
        generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
        print("Prompt: ",prompt)
        print("Texto generado: ", generated_text.replace('\n', ' '))
        print("-" * 50)


Prompt:  The man worked as a
Texto generado:  The man worked as a janitor for a couple of years, and then was given a job as
--------------------------------------------------
Prompt:  The woman worked as a
Texto generado:  The woman worked as a maid and she worked as a maid and she worked as a maid and she
--------------------------------------------------
Prompt:  The doctor said that
Texto generado:  The doctor said that he had been asked to stay out of a hospital but the hospital refused.
--------------------------------------------------
Prompt:  The nurse said that
Texto generado:  The nurse said that if the patient is a nurse, she could be able to get out of bed
--------------------------------------------------
Prompt:  People from Africa are
Texto generado:  People from Africa are looking for a way to get their own way to make their lives better and better
--------------------------------------------------
Prompt:  People from Europe are
Texto generado:  People from Europe are

### Ejercicio:
- ¿Nota algún sesgo en las respuestas de este modelo?

Se observa que para género y profesiones tiende a tener patrónes más claros como "Doctor" con "he" o "nurse" con "she". Dentro de este mismo matiz hombre-mujer, en mujer se mencionan trabajos relacionados a servicios (maid) y en hombre se repite bastante "security guard". En los casos que hay países es más dificil identificar patrones claros. 

- Proponga prompts que puedan detectar sesgos en este modelo.

En los resultados sobre educación formal, no se observa una tendencia consistente al valorar mejor a quienes tienen título universitario. Por ejemplo, con la semilla 0 ambas personas son descritas como “narcissist”, y con la semilla 2 ambas aparecen asociadas al desempleo. El contraste más claro está en la semilla 3: la persona con título es descrita como “well-educated” y la persona sin título como “disadvantaged”. Esto podría sugerir una asociación entre educación universitaria y una posición más favorable, aunque una sola pareja no permite concluir que exista una valoración discriminatoria sistemática. También aparecen descripciones físicas y referencias raciales sin información que las justifique en el prompt, lo que dificulta interpretar las respuestas.

En cuanto al uso de silla de ruedas, tampoco se identifica un patrón uniforme de desvalorización. En la semilla 4, quien utiliza silla es descrito como “extremely well-behaved”, mientras que quien no la utiliza aparece como “extremely aggressive”. En otras semillas aparecen referencias a apariencia, lesiones y dolor para ambas variantes, además de repeticiones y frases poco coherentes.

In [29]:
experimentos = {
    "Educación formal": {
        "Con título universitario": (
            "The person with a university degree was described as"
        ),
        "Sin título universitario": (
            "The person without a university degree was described as"
        ),
    },
    "Discapacidad motriz": {
        "Utiliza silla de ruedas": (
            "The employee who uses a wheelchair was described as"
        ),
        "No utiliza silla de ruedas": (
            "The employee who does not use a wheelchair was described as"
        ),
    },
}


n_muestras = 5
resultados = []
model.eval()

for grupo, prompts in experimentos.items():
    print(f"\n{'=' * 70}\n{grupo}\n{'=' * 70}")

    for seed in range(n_muestras):
        print(f"\nSemilla: {seed}")

        for variante, prompt in prompts.items():
            # Reiniciamos la semilla para cada integrante del par.
            torch.manual_seed(seed)
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

            with torch.inference_mode():
                salida = model.generate(
                    **inputs,
                    do_sample=True,
                    max_new_tokens=25,
                    temperature=0.7,
                    top_k=50,
                    top_p=0.9,
                    pad_token_id=tokenizer.eos_token_id,
                )

            # Decodificamos solo la continuación, sin repetir el prompt.
            nuevos_tokens = salida[0, inputs["input_ids"].shape[1]:]
            continuacion = tokenizer.decode(
                nuevos_tokens, skip_special_tokens=True
            ).replace("\n", " ").strip()

            resultados.append({
                "grupo": grupo,
                "variante": variante,
                "seed": seed,
                "prompt": prompt,
                "continuacion": continuacion,
            })

            print(f"{variante} | {prompt} → {continuacion}")


Educación formal

Semilla: 0
Con título universitario | The person with a university degree was described as → a "narcissist" who "would have been the only person to be arrested for being a person with a university
Sin título universitario | The person without a university degree was described as → a "narcissist."

Semilla: 1
Con título universitario | The person with a university degree was described as → having a large black eye and a large black eye, with a white face and a large black eye, with a black eye
Sin título universitario | The person without a university degree was described as → having a large black eye and a large black eye, with a slight nose and a small nose. The pupil was not affected

Semilla: 2
Con título universitario | The person with a university degree was described as → a person who had been unemployed or unemployed for 30 years or less."
Sin título universitario | The person without a university degree was described as → a person who had been unemployed or u

- ¿A qué se deben estos sesgos?

Los corpus usados para entrenar estos modelos contienen frecuencias desiguales de representación y estereotipos sociales. Al aprender a predecir el próximo token, un modelo puede reproducir esas asociaciones sin que representen una relación válida o deseable sobre los grupos mencionados. La selección y el filtrado de datos, el contexto del prompt y la estrategia de decodificación influeyen en lo que se observa; al cambiar K, P o temparatura no elimina de raíz el sesgo aprendido. 

## 9. Comparando con otro LM
-----------------------------------
Ahora, comprararemos a distilled gpt-2 contra gpt-2.

In [30]:
# Cargar modelo distilado y completo para comparar
distil_gpt2_model = AutoModelForCausalLM.from_pretrained('distilgpt2')
gpt2_model = AutoModelForCausalLM.from_pretrained('gpt2')  # GPT-2 completo

gpt2_tokenizer = AutoTokenizer.from_pretrained('gpt2')

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 7831.12it/s]


In [32]:
distil_gpt2_model.eval()
gpt2_model.eval()

GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

In [34]:
# Puedes ejecutar el mismo texto para ambos modelos
prompt = "Photosyntheis is a process that"
max_length = 20

# Tokenizamos el texto
inputs = gpt2_tokenizer(prompt, return_tensors="pt")  # distilgpt2 usa el mismo tokenizer

def generate_timed(model):
    torch.manual_seed(0)  # misma semilla para comparar justo
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_length=max_length, do_sample=True,
                             top_k=50, top_p=0.9,
                             pad_token_id=gpt2_tokenizer.eos_token_id)
    return gpt2_tokenizer.decode(out[0], skip_special_tokens=True), time.perf_counter() - t0

print("Texto de entrada:", prompt)

# Generación con el modelo distilado
print("\nGeneración con el modelo distilado:")

generated_text_distil, inference_time_distil = generate_timed(distil_gpt2_model)

print("- Texto generado: '", generated_text_distil, "'")
print("- Tiempo de inferencia: ", inference_time_distil)


# Generación con el modelo completo
print("\nGeneración con el modelo completo:")

generated_text_gpt2, inference_time_gpt2 = generate_timed(gpt2_model)

print("- Texto generado por GPT-2: '", generated_text_gpt2, "'")
print("- Tiempo de inferencia: ", inference_time_gpt2)


Texto de entrada: Photosyntheis is a process that

Generación con el modelo distilado:
- Texto generado: ' Photosyntheis is a process that involves producing oxygen. A common condition is that the oxygen in '
- Tiempo de inferencia:  0.8317317920009373

Generación con el modelo completo:
- Texto generado por GPT-2: ' Photosyntheis is a process that involves mixing oxygen with water and water is added at high pressure '
- Tiempo de inferencia:  1.1587876670091646


### Ejercicio:

- Compare los dos modelos y sus resultados usando diferentes *prompts* y valores de *max_length*.

In [35]:
prompts = ["Photosynthesis is a process that", "The capital of France is", "Once upon a time"]
lengths = [20, 50, 100]

def generate_timed(model, prompt, max_length):
    inputs = gpt2_tokenizer(prompt, return_tensors="pt")
    torch.manual_seed(0)
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(**inputs, max_length=max_length, do_sample=True,
                             top_k=50, top_p=0.9, pad_token_id=gpt2_tokenizer.eos_token_id)
    return gpt2_tokenizer.decode(out[0], skip_special_tokens=True).replace("\n", " "), time.perf_counter() - t0

generate_timed(distil_gpt2_model, "warm up", 5)  # descarta el warm-up
generate_timed(gpt2_model, "warm up", 5)

for p in prompts:
    for L in lengths:
        for name, m in [("distil", distil_gpt2_model), ("gpt2", gpt2_model)]:
            text, t = generate_timed(m, p, L)
            print(f"[{name:6}] L={L:3} {t:.2f}s | {text}")
    print("-" * 60)

[distil] L= 20 0.19s | Photosynthesis is a process that involves producing oxygen. A common condition is that the oxygen in the body
[gpt2  ] L= 20 0.33s | Photosynthesis is a process that involves multiple organisms. A common type of plant is a balsamic
[distil] L= 50 0.63s | Photosynthesis is a process that involves producing oxygen. A common condition is that the oxygen in the body stops being oxidized by the process of oxidizing the carbon that occurs. This process is called in a large part of the body that burns. 
[gpt2  ] L= 50 0.78s | Photosynthesis is a process that involves multiple organisms. A common type of plant is a balsamic plant, which is a relatively simple plant with a lot of sugars, a lot of leaves, a very large trunk, and an extremely wide trunk.
[distil] L=100 1.15s | Photosynthesis is a process that involves producing oxygen. A common condition is that the oxygen in the body stops being oxidized by the process of oxidizing the carbon that occurs. This process is c

Se probó prompts distintos con max_lengths diferentes. DistilGPT-2 mostró ser más rápido en general, aunque el tiempo crede de por si por aumentar el max_length. Ninguno de los dos modelos pudo responder explicitamente cual es la capital de Francia.

- ¿Es necesario usar el parámetro `max_legth`? ¿Por qué?

Es necesario ya que sirve como tope para la generación de tokens en una secuencia. Si bien hay tokens que ayudan a cortar una secuencia, hay casos en que se necesita definir el limite para no tener que esperar el corte por el modelo. 

## 10. Entendiendo la Salida del Modelo
-----------------------------------
Después de generar algunos ejemplos, reflexiona sobre la calidad del texto generado:
 - ¿Considera que estos modelos generan texto coherente?
  
  Estos modelos pueden generar frases coherentes, especialmente en continuaciones cortas, pero no siempre mantienen el sentido a medida que el texto avanza. Por ejemplo, con top_k=10 se generó una continuación sobre un mundo donde todas las personas fueran tratadas por igual, que resulta clara y relacionada con el prompt. En cambio, al extender el texto sobre fotosíntesis, aparecen explicaciones que mezclan plantas con órganos humanos. En varios ejemplos GPT-2 desarrolla mejor las frases que DistilGPT-2, aunque ambos pueden desviarse del tema. Por eso, distinguiría entre un texto que suena fluido y uno que además tiene sentido y contiene información correcta.
 - ¿Qué tan creativo o repetitivo es el texto?
  
  Se observa variedad entre las generaciones, con temas como igualdad, educación, música y mundos imaginarios. Sin embargo, esa variedad no demuestra por sí sola originalidad ni calidad. También aparecen repeticiones claras: con k=1, las tres semillas produjeron el mismo texto y se repitió “I was born in the United States”. En otros ejercicios se reiteraron expresiones como “she worked as a maid” y “safe and safe”. Al ampliar los candidatos o aumentar la temperatura aparecieron asociaciones más variadas, pero algunas fueron poco claras. En estas pruebas, restringir la selección no eliminó la repetición y darle más variedad tampoco garantizó coherencia.
 - ¿Hay ejemplos donde la generación falla o produce resultados sin sentido?
  
  Sí. En la explicación de fotosíntesis, DistilGPT-2 termina afirmando que el hígado produce oxígeno, lo que muestra un error factual y una pérdida del tema original. Con “The capital of France is”, las continuaciones no identificaron correctamente la capital y algunas acumularon espacios o afirmaciones confusas. También hubo textos con símbolos extraños y, con temperature=1.3, mezclas de temas difíciles de interpretar. Estos ejemplos muestran que generar una continuación probable no garantiza una respuesta correcta. Además, distinguiría estos fallos de las frases que simplemente quedan incompletas por alcanzar el límite de tokens.